# Notebook 03: Model Training

Runs GWR → appends local coefficients → trains calibrated LightGBM → builds SHAP explainer. All artefacts are persisted to `models/`.

In [ ]:
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print("Current Working Directory:", os.getcwd())


Current Working Directory: c:\Users\adity\OneDrive\ドキュメント\X-Pand\X-Pand


## Load data

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import numpy as np
import joblib
import geopandas as gpd

gdf = gpd.read_file('../data/processed/grid.geojson')
X = joblib.load('../data/processed/X_train.pkl')
y = joblib.load('../data/processed/y_train.pkl')

# Determine feature names from the features saved in notebook 02
from src.feature_engineering import FEATURE_COLUMNS
feature_names = list(FEATURE_COLUMNS)

print(f"Grid cells: {len(gdf)}")
print(f"X shape: {X.shape}, y shape: {y.shape}")
print(f"Feature names: {feature_names}")

Grid cells: 12987
X shape: (12987, 13), y shape: (12987,)
Feature names: ['pop_density', 'pop_density_log', 'competitor_count', 'competitor_density_1km', 'nearest_competitor_km', 'market_saturation', 'opportunity_gap', 'income_index', 'road_density', 'transit_stops', 'warehouse_proximity_km', 'lag_profitable', 'lag_pop_density']


## Run GWR (Geographically Weighted Regression)

Fits GWR (or falls back to OLS if GWR times out) and extracts local coefficients as new features.

In [ ]:
from src.gwr_model import run_gwr, extract_gwr_features, save_gwr_coeffs

os.makedirs('../models', exist_ok=True)

gwr_dict, gwr_results = run_gwr(gdf, X, y)

# Print summary
if hasattr(gwr_results, 'bw'):
    print(f"GWR bandwidth selected: {gwr_results.bw}")
else:
    print("GWR bandwidth: N/A (OLS fallback)")

mean_r2 = float(gwr_results.localR2.mean())
print(f"Mean local R²: {mean_r2:.4f}")

gdf = extract_gwr_features(gdf, gwr_results)
save_gwr_coeffs(gwr_results, '../models/gwr_coeffs.pkl')
print("GWR coefficients saved.")

## Add GWR features to X

Appends `gwr_intercept` and `gwr_local_r2` to the feature matrix so the LightGBM model can leverage geographically varying information.

In [ ]:
gwr_features = gdf[['gwr_intercept', 'gwr_local_r2']].values.astype(np.float64)
X = np.hstack([X, gwr_features])
feature_names = feature_names + ['gwr_intercept', 'gwr_local_r2']

print(f"Augmented X shape: {X.shape}")
print(f"Updated feature names ({len(feature_names)}): {feature_names}")

## Train LightGBM classifier

SMOTE oversampling → 5-fold stratified CV → isotonic calibration.

In [ ]:
from src.lgbm_model import train_lgbm, save_model

model, cv_scores = train_lgbm(X, y, feature_names)

print("\n--- Cross-Validation Results ---")
for fold, score in cv_scores.items():
    print(f"  Fold {fold}: F1 = {score:.4f}")

mean_f1 = sum(cv_scores.values()) / len(cv_scores)
print(f"\nMean F1: {mean_f1:.4f}")

if mean_f1 < 0.8:
    print("WARNING: Mean F1 below 0.8 target!")
else:
    print("✓ F1 target met.")

save_model(model, '../models/lgbm_model.pkl')

# Also re-save the augmented features for the API
import pandas as pd
aug_features_df = gdf[['grid_id']].copy()
for i, fname in enumerate(feature_names):
    aug_features_df[fname] = X[:, i]
aug_features_df.to_pickle('../data/processed/features.pkl')
print("Updated features.pkl with GWR-augmented features.")

## Build SHAP explainer

Creates a TreeExplainer from the calibrated LightGBM model and generates a global summary plot.

In [ ]:
from src.explainer import build_shap_explainer, save_explainer
import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

explainer, shap_values = build_shap_explainer(model, X)
save_explainer(explainer, '../models/shap_explainer.pkl')

print(f"SHAP values shape: {shap_values.shape}")
print(f"Top features by mean |SHAP|:")
mean_abs = np.abs(shap_values).mean(axis=0)
for idx in np.argsort(mean_abs)[::-1][:5]:
    print(f"  {feature_names[idx]}: {mean_abs[idx]:.6f}")

# Summary plot
shap.summary_plot(
    shap_values, X,
    feature_names=feature_names,
    max_display=10,
    show=False,
)
plt.tight_layout()
plt.show()

## Save updated grid with predictions

Re-save the grid GeoJSON so notebook 04 and the API can use it.

In [ ]:
# Add p_profit column using the calibrated model
p_profit = model.predict_proba(X)[:, 1]
gdf['p_profit'] = p_profit
gdf.to_file('../data/processed/grid.geojson', driver='GeoJSON')

print(f"Grid saved with p_profit column (mean={p_profit.mean():.4f}).")
print("\nAll training artefacts saved to models/:")
for f in os.listdir('../models'):
    size_kb = os.path.getsize(os.path.join('../models', f)) / 1024
    print(f"  {f} ({size_kb:.1f} KB)")